In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
import torch.ao.quantization as quant
import types
import torch.ao.quantization as aq
from torchvision.models.resnet import BasicBlock
from torch.ao.quantization import get_default_qat_qconfig, prepare_qat_fx, convert_fx

import os
import logging
from datetime import datetime

In [2]:
os.makedirs("./data", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./trained_models", exist_ok=True)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=2)

## Full ResNet18 Traininig

In [5]:
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

/home/bohdan/RAI/rai-env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bohdan/RAI/rai-env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
0.3%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/bohdan/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


In [6]:
def training_loop(model, model_name, trainloader, testloader):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    num_epochs = 1

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

In [7]:
start_of_training_timestamp = training_loop(model, "resnet18_cifar", trainloader, testloader)

2025-10-03 15:18:08,193 [INFO] Epoch [1/1] Train Loss: 1.0316, Train Acc: 64.46% Test Loss: 0.8402, Test Acc: 71.55%


In [8]:
model_path = f"./trained_models/resnet18_cifar10_{start_of_training_timestamp}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

Model saved as ./trained_models/resnet18_cifar10_03.10.2025-15:17:55.pth
Model size: 42.73 MB


## QAT ResNet Training

In [ ]:
model = resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 10)

In [ ]:
class QuantizableBasicBlock(nn.Module):
    def __init__(self, orig_block: BasicBlock):
        super().__init__()
        # reuse original submodules (share parameters)
        self.conv1 = orig_block.conv1
        self.bn1 = orig_block.bn1
        self.relu = orig_block.relu
        self.conv2 = orig_block.conv2
        self.bn2 = orig_block.bn2
        self.downsample = orig_block.downsample  # may be None
        self.stride = orig_block.stride

        # quant/dequant stubs for local quantization boundary
        self.quant = quant.QuantStub()
        self.dequant = quant.DeQuantStub()

    def forward(self, x):
        identity = x

        # Quantize input to block -> convs will be observed/quantized
        out = self.quant(x)

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Dequantize before adding the residual so addition runs in float
        out = self.dequant(out)

        if self.downsample is not None:
            # downsample usually contains conv+bn; keep it in float (we assume not quantized)
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

In [ ]:
def make_blocks_quantizable(model):
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            orig_block = layer[i]
            # wrap
            qblock = QuantizableBasicBlock(orig_block)
            layer[i] = qblock  # inplace replacement

make_blocks_quantizable(model)

In [ ]:
try:
    # Fuse conv1 + bn1 + relu (top)
    quant.fuse_modules(model, [['conv1', 'bn1', 'relu']], inplace=True)
except Exception as e:
    print("Warning: top-level fuse failed:", e)

# For each wrapped BasicBlock
for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(model, layer_name)
    for i in range(len(layer)):
        block = layer[i]
        # fuse conv1+bn1+relu and conv2+bn2 inside the wrapped block
        # note: block.relus exist as attribute `relu`
        try:
            quant.fuse_modules(block, [['conv1', 'bn1', 'relu'], ['conv2', 'bn2']], inplace=True)
        except Exception as e:
            # If fuse fails, keep going (but fusing is recommended)
            print(f"Warning: fuse failed for {layer_name}.{i}: {e}")

In [ ]:
default_qconfig = quant.get_default_qat_qconfig('fbgemm')
model.qconfig = default_qconfig  # global default

# Keep first conv and final fc in float (no qconfig)
model.conv1.qconfig = None
model.fc.qconfig = None

# Also ensure any downsample modules (projections) are kept float (if present)
for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(model, layer_name)
    for block in layer:
        if getattr(block, 'downsample', None) is not None:
            # downsample is typically nn.Sequential(conv, bn)
            block.downsample.qconfig = None

In [ ]:
qconfig = quant.get_default_qat_qconfig("fbgemm")
def apply_qconfig(model, qconfig):
    for name, module in model.named_children():
        if isinstance(module, nn.Conv2d) and name == "conv1":
            module.qconfig = None  # keep FP32
        elif isinstance(module, nn.Linear) and name == "fc":
            module.qconfig = None  # keep FP32
        elif isinstance(module, nn.ReLU) or isinstance(module, nn.BatchNorm2d):
            module.qconfig = None  # leave non-critical ops FP32
        else:
            module.qconfig = qconfig  # quantize residual conv blocks
        apply_qconfig(module, qconfig)
apply_qconfig(model, qconfig)

In [ ]:
training_loop()
model.eval()
quantized_model = quant.convert_fx(model)